# Step 2.5 — Manual Corpus Curation (Key Reference Works)

**Purpose:** Before bulk-downloading 100–200 volumes in Step 3a, identify the highest-value Internet Archive sources first.

This notebook:

1. **Queries Internet Archive full-text search** (via `archive.org/fulltext/`) for each person in the database, surfacing volumes that contain their name in OCR'd body text.
2. **Falls back to metadata search** (`advancedsearch.php`) when full-text search is unavailable or returns no results.
3. **Searches for priority reference works** (Appleton's, National Cyclopædia, Poor's Manual, press trade journals, biographical directories) that are almost certain to contain entries for our persons.
4. **Ranks volumes** by how many persons in the dataset they mention — high coverage = high download priority.
5. **Outputs `ia_corpus_manifest.json`** — a structured list of IA identifiers that Step 3a will use to drive bulk download.

After the automated cells run, **manually review and edit `ia_corpus_manifest.json`** before proceeding to Step 3a:
- Add missing IA identifiers (specific Appleton's volumes, Poor's Manual annual editions, county histories)
- Remove obviously irrelevant items (false positives on common names)
- Promote important items to `priority: high`

In [1]:
import sys, json, time, re
from pathlib import Path
import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry
from tqdm import tqdm

sys.path.insert(0, str(Path('.')))
from db import get_connection, log_source, get_all_persons, name_variants_for

conn = get_connection()
MANIFEST_PATH = Path('ia_corpus_manifest.json')  # output: curated list of IA item IDs

# --- Robust HTTP session with retry logic ---
def make_session(retries=3, backoff_factor=1.5, status_forcelist=(429, 500, 502, 503, 504)):
    session = requests.Session()
    retry = Retry(
        total=retries,
        read=retries,
        connect=retries,
        backoff_factor=backoff_factor,
        status_forcelist=status_forcelist,
        allowed_methods=['GET'],
    )
    adapter = HTTPAdapter(max_retries=retry)
    session.mount('https://', adapter)
    session.mount('http://', adapter)
    session.headers.update({'User-Agent': 'RailroadTiesPipeline/1.0'})
    return session

http = make_session()

# Base delay between IA requests (seconds)
IA_REQUEST_DELAY = 1.5

## Priority reference works to check manually

Before running the automated search, verify these high-value volumes are available on Internet Archive:

| Category | Title | IA search term |
|---|---|---|
| National biography | Appleton's Cyclopædia of American Biography (1887–89) | `appleton cyclopaedia american biography` |
| National biography | National Cyclopædia of American Biography (1892+) | `national cyclopaedia american biography` |
| Press history | Frederic Hudson, Journalism in the United States (1873) | `hudson journalism united states 1873` |
| Press trade journal | The Journalist (1883–1907) | `"the journalist" newspaper weekly` |
| Press trade journal | The Fourth Estate (1894+) | `"fourth estate" newspaper journalism` |
| Biographical directory | Who's Who in America (1899+) | `"who's who in america"` |
| National biography | Dictionary of American Biography (DAB) | `"dictionary of american biography"` |
| Railroad finance | Poor's Manual of Railroads (annual, 1868+) | `poor manual railroads` |
| Congressional | Congressional Directory / Biographical Directory of Congress | `congressional directory biographical` |
| State press assoc. | State press association proceedings (per state) | `"[State] press association" proceedings` |
| County histories | Search per state, e.g. 'History of [County] County [State]' | varies |

The cells below search Internet Archive for each of these and for your persons' names. After reviewing the results, edit the `ia_corpus_manifest.json` file to add/remove items before running Step 3a.

## Internet Archive search helpers

Two search strategies:
1. **Full-text search** via `archive.org/fulltext/` — searches inside OCR'd page text. Best for finding incidental mentions of a person inside a book.
2. **Metadata search** via `advancedsearch.php` — searches title, description, subject, creator fields. Best for finding known reference works by title.

We use full-text search for person-name queries and metadata search for reference-work discovery.

In [7]:
def search_ia_fulltext(query, rows=20, scope='all'):
    """
    Search Internet Archive's full-text search index (FTS).
    This searches *inside* OCR'd text of books, not just metadata.
    
    Uses the scrape API with the '!L' prefix, which is how IA's own
    `internetarchive` Python library triggers full-text search.
    
    Endpoint: https://be-api.us.archive.org/ia-pub-fts-api
    
    Parameters
    ----------
    query : str
        Search query (supports quoted phrases).
    rows : int
        Max results to return.
    scope : str
        'all' for everything, or an IA collection identifier.
    
    Returns
    -------
    list of dicts with keys: identifier, title, date, fields, highlight
    """
    params = {
        'q': query,
        'output': 'json',
        'rows': rows,
        'scope': scope,
    }
    
    try:
        resp = http.get(
            'https://be-api.us.archive.org/ia-pub-fts-api',
            params=params,
            timeout=60,
        )
        resp.raise_for_status()
        data = resp.json()
        
        results = []
        for hit in data.get('hits', {}).get('hits', []):
            src = hit.get('_source', hit.get('fields', {}))
            highlight = hit.get('highlight', {}).get('text', [''])
            results.append({
                'identifier': hit.get('_id', src.get('identifier', '')),
                'title': src.get('title', ''),
                'date': src.get('date', src.get('year', '')),
                'creator': src.get('creator', ''),
                'description': '',
                'subject': src.get('subject', []),
                'highlight': highlight[0][:300] if highlight else '',
                'search_type': 'fulltext',
            })
        return results
    
    except Exception as e:
        print(f"  FTS error for '{query}': {e}")
        return []


def search_ia_metadata(query, rows=20, date_range=None):
    """
    Search Internet Archive metadata fields (title, description, subject, creator).
    Endpoint: https://archive.org/advancedsearch.php
    
    Use this for discovering known reference works by title/subject.
    """
    params = {
        'q': query,
        'fl[]': ['identifier', 'title', 'date', 'description', 'subject', 'creator'],
        'rows': rows,
        'page': 1,
        'output': 'json',
    }
    if date_range:
        params['q'] += f' AND date:[{date_range[0]} TO {date_range[1]}]'
    
    try:
        resp = http.get(
            'https://archive.org/advancedsearch.php',
            params=params,
            timeout=30,
        )
        resp.raise_for_status()
        data = resp.json()
        docs = data.get('response', {}).get('docs', [])
        return [{
            'identifier': d.get('identifier', ''),
            'title': d.get('title', ''),
            'date': d.get('date', ''),
            'description': str(d.get('description', ''))[:200],
            'subject': d.get('subject', []),
            'creator': d.get('creator', ''),
            'highlight': '',
            'search_type': 'metadata',
        } for d in docs]
    except Exception as e:
        print(f"  Metadata search error for '{query}': {e}")
        return []


def search_ia_for_person(name, variants=None, first_yr=None, last_yr=None):
    """
    Search IA for a person using full-text search, with name variants.
    Falls back to metadata search if FTS returns nothing.
    Deduplicates results by IA identifier.
    """
    all_results = []
    seen_ids = set()
    
    search_names = [name]
    if variants:
        # Add top variants, limit to avoid too many queries
        search_names.extend(variants[:2])
    
    for search_name in search_names:
        # Full-text search (searches inside book text)
        fts_results = search_ia_fulltext(f'"{search_name}"', rows=8)
        for r in fts_results:
            if r['identifier'] and r['identifier'] not in seen_ids:
                seen_ids.add(r['identifier'])
                all_results.append(r)
        time.sleep(IA_REQUEST_DELAY)
    
    # If FTS returned very little, supplement with metadata search
    if len(all_results) < 3:
        query = f'"{name}" mediatype:texts'
        if first_yr and last_yr:
            query += f' AND date:[{max(1860, (first_yr or 1870)-5)} TO {(last_yr or 1900)+20}]'
        meta_results = search_ia_metadata(query, rows=10)
        for r in meta_results:
            if r['identifier'] and r['identifier'] not in seen_ids:
                seen_ids.add(r['identifier'])
                all_results.append(r)
        time.sleep(IA_REQUEST_DELAY)
    
    return all_results

## Search for priority reference works

These are searched via **metadata search** since we know the titles/subjects.

In [3]:
PRIORITY_SEARCHES = [
    # --- National / general biographical references ---
    ("appleton cyclopaedia american biography",
     "Appleton's Cyclopædia of American Biography", ("1886", "1900")),
    ("national cyclopaedia american biography",
     "National Cyclopædia of American Biography", ("1891", "1920")),
    ('"dictionary of american biography"',
     "Dictionary of American Biography (DAB)", ("1928", "1945")),
    ('"who\'s who in america"',
     "Who's Who in America", ("1899", "1920")),
    
    # --- Press history & trade journals ---
    ("hudson journalism united states 1873",
     "Frederic Hudson - Journalism in the United States", ("1870", "1880")),
    ('"the journalist" newspaper weekly trade',
     "The Journalist (trade weekly, 1883–1907)", ("1883", "1907")),
    ('"fourth estate" newspaper journalism trade',
     "The Fourth Estate (trade journal, 1894+)", ("1894", "1920")),
    ("biographical history newspaper editor publisher",
     "Biographical newspaper/press histories", ("1860", "1920")),
    ("history journalism press newspaper 1880",
     "Press/journalism histories", ("1860", "1920")),
    
    # --- Railroad finance ---
    ("poor manual railroads",
     "Poor's Manual of Railroads", ("1868", "1920")),
    
    # --- Congressional ---
    ('"congressional directory" biographical',
     "Congressional Directory (biographical)", ("1860", "1920")),
    ('"biographical directory" congress',
     "Biographical Directory of Congress", ("1860", "1930")),
]

priority_results = {}
for query, label, date_range in PRIORITY_SEARCHES:
    print(f"\nSearching: {label}")
    results = search_ia_metadata(query, rows=15, date_range=date_range)
    priority_results[label] = results
    for r in results[:5]:
        print(f"  [{r['identifier']}] {r['title'][:70]} ({r['date']})")
    if not results:
        print("  (no results — try searching archive.org manually)")
    time.sleep(IA_REQUEST_DELAY)


Searching: Appleton's Cyclopædia of American Biography
  [americanbio05wilsuoft] Appletons' cyclopædia of American biography Volume 5 (1891-01-01T00:00:00Z)
  [bib_fict_1073745_6] Appletons’ cyclopædia of American biography edited by James Grant Wils (1889-01-01T00:00:00Z)
  [appletonscyclop01wils] Appletons' cyclopædia of American biography (1887-01-01T00:00:00Z)
  [appletonscyclopa04wils] Appleton's cyclopaedia of American biography (1900-01-01T00:00:00Z)
  [appletonscyclopa06wilsuoft] Appleton's cyclopaedia of American biography (1887-01-01T00:00:00Z)

Searching: National Cyclopædia of American Biography
  [cu31924020334813] The National cyclopædia of American biography : being the history of t (1893-01-01T00:00:00Z)
  [nationalcyclopae08newy] The National cyclopaedia of American biography, being the history of t (1893-08-19T00:00:00Z)
  [nationalcyclopae00newy] The National cyclopaedia of American biography, being the history of t (1893-08-19T00:00:00Z)
  [cu31924020334748] The Na

## Search IA for each person (full-text + name variants)

In [8]:
# Search IA for all persons using full-text search + name variants
persons = get_all_persons(conn)
print(f"Searching IA for {len(persons)} persons (full-text + variants)...")
print(f"Estimated time: ~{len(persons) * 3 * IA_REQUEST_DELAY / 60:.0f} minutes\n")

ia_hits_per_person = {}
for person in tqdm(persons):
    rid = person['research_id']
    name = person['canonical_name']
    
    # Generate name variants for broader coverage
    try:
        variants = name_variants_for(name)
    except Exception:
        variants = []
    
    # sqlite3.Row doesn't support .get(); use dict() or direct indexing
    p = dict(person)
    results = search_ia_for_person(
        name,
        variants=variants,
        first_yr=p.get('first_active_year'),
        last_yr=p.get('last_active_year'),
    )
    
    ia_hits_per_person[rid] = results
    
    for r in results:
        if r['identifier']:
            log_source(
                conn, rid, 'internet_archive',
                title=r['title'],
                url=f"https://archive.org/details/{r['identifier']}",
                item_id=r['identifier'],
                snippet=r.get('highlight', r.get('description', ''))[:500],
                relevance_note=(
                    f"search_type: {r['search_type']}; "
                    f"creator: {r.get('creator', '')}; "
                    f"date: {r.get('date', '')}"
                ),
                discovery_step='2.5_ia_search',
            )

print(f"\nDone. Total IA hits: {sum(len(v) for v in ia_hits_per_person.values())}")
print(f"Persons with ≥1 hit: {sum(1 for v in ia_hits_per_person.values() if v)}/{len(persons)}")

Searching IA for 570 persons (full-text + variants)...
Estimated time: ~43 minutes



100%|██████████| 570/570 [32:50<00:00,  3.46s/it]


Done. Total IA hits: 5382
Persons with ≥1 hit: 556/570


## Identify high-value volumes (mentioned by 2+ persons)

In [10]:
multi_person_volumes = conn.execute("""
    SELECT sd.item_id, sd.title, sd.url,
           COUNT(DISTINCT sd.research_id) as persons_count,
           GROUP_CONCAT(p.canonical_name, ' | ') as person_names
    FROM (
        SELECT DISTINCT item_id, title, url, research_id
        FROM sources_discovered
        WHERE source_type = 'internet_archive'
          AND item_id IS NOT NULL AND item_id != ''
    ) sd
    JOIN persons p ON sd.research_id = p.research_id
    GROUP BY sd.item_id
    HAVING persons_count >= 2
    ORDER BY persons_count DESC
    LIMIT 50
""").fetchall()

print(f"Volumes mentioning 2+ persons in dataset ({len(multi_person_volumes)} found):\n")
print(f"{'Persons':>7}  {'IA ID':<40} {'Title'}")
print('-' * 90)
for r in multi_person_volumes:
    print(f"  {r['persons_count']:5d}  {r['item_id']:<40} {(r['title'] or '')[:45]}")

Volumes mentioning 2+ persons in dataset (50 found):

Persons  IA ID                                    Title
------------------------------------------------------------------------------------------
      5  biennialreportka00kans|4c3b8b98fc821419ffed85df4dd367620d600e29 
      4  guidetowisconsin00oehl|1687662bd180c58b39bf54fb57f31dc60677d89a 
      4  guidetowisconsin00iloehl|c6bb27f842df961e68b587b67ebd66ea6660ec1d 
      4  frontiergovernor0000plum|06126bd5ee144d224a5142ccf3a1c34d20bf11e2 
      4  cu31924028871552|a5cf0af2715570fca07790eab38a4bbdf814638c 
      4  annalsofkansas00wild|502b3c3b02481f8e77f214cdcf96e7b9a7207d70 
      4  annalsofemporial00stot|4cb670c73af161ac9f64c3bd41311a76d54bcff2 
      3  wisconsin-rapids-daily-tribune-1928-07-26|4a7324b884208488dffce234de662d8baf3c48b1 
      3  westminsterwayit0000port|3aa09ca8bdd5a3cfa97ae3108b3b08005f12f7e4 
      3  thirteenthseedan1888jmph|18c6ec713528df09335eabbca5d23f7808e8cc33 
      3  swordsofspain0000unse|11f309ef4

## Build and save the corpus manifest

Review the volumes above, then run the cell below to save the manifest. You can manually edit `ia_corpus_manifest.json` after running — add IA identifiers from the priority reference works above, remove low-relevance items, and add notes.

In [11]:
# Build manifest from high-value volumes
manifest = {
    'description': 'Curated list of Internet Archive items for Step 3a bulk download',
    'created': __import__('datetime').datetime.utcnow().isoformat(),
    'search_notes': {
        'fulltext_endpoint': 'https://archive.org/fulltext/',
        'metadata_endpoint': 'https://archive.org/advancedsearch.php',
        'name_variants_used': True,
    },
    'items': [],
}

seen_ids = set()

# Add multi-person volumes (auto-ranked)
for r in multi_person_volumes:
    iid = r['item_id']
    if iid in seen_ids:
        continue
    seen_ids.add(iid)
    manifest['items'].append({
        'identifier': iid,
        'title': r['title'] or '',
        'url': r['url'] or f'https://archive.org/details/{iid}',
        'persons_count': r['persons_count'],
        'person_names_sample': (r['person_names'] or '')[:200],
        'priority': 'high' if r['persons_count'] >= 5 else 'medium',
        'notes': '',
        'added_by': 'auto_step2_5',
    })

# Add priority reference works (if found above)
for label, results in priority_results.items():
    for r in results[:3]:
        iid = r['identifier']
        if not iid or iid in seen_ids:
            continue
        seen_ids.add(iid)
        manifest['items'].append({
            'identifier': iid,
            'title': r['title'],
            'url': f'https://archive.org/details/{iid}',
            'persons_count': 0,
            'person_names_sample': '',
            'priority': 'high',
            'notes': f'Priority reference: {label}',
            'added_by': 'auto_step2_5_priority',
        })

# Save
with open(MANIFEST_PATH, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f"Manifest saved to {MANIFEST_PATH}")
print(f"  Total items: {len(manifest['items'])}")
print(f"  High priority: {sum(1 for i in manifest['items'] if i['priority'] == 'high')}")
print(f"  Medium priority: {sum(1 for i in manifest['items'] if i['priority'] == 'medium')}")
print()
print('NEXT STEP: Review and edit ia_corpus_manifest.json manually.')
print('  - Add any missing IA identifiers (e.g. specific Appleton\'s volumes, Poor\'s Manual years)')
print('  - Remove low-relevance items')
print('  - Set priority="high" for the most important volumes')
print('Then run step3a_internet_archive.ipynb')

Manifest saved to ia_corpus_manifest.json
  Total items: 76
  High priority: 27
  Medium priority: 49

NEXT STEP: Review and edit ia_corpus_manifest.json manually.
  - Add any missing IA identifiers (e.g. specific Appleton's volumes, Poor's Manual years)
  - Remove low-relevance items
  - Set priority="high" for the most important volumes
Then run step3a_internet_archive.ipynb


C:\Users\samwt\AppData\Local\Temp\ipykernel_11980\1995371465.py:4: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  'created': __import__('datetime').datetime.utcnow().isoformat(),


## State-specific sources: mug books + press association proceedings

For each state in your dataset, search Internet Archive for:
1. **County and state biographical histories ('mug books')** from the 1880s–90s — newspaper owners are disproportionately well represented because they paid to be included.
2. **State press association proceedings** — annual publications with biographical sketches of editor/publisher members.

Run the cell below to search for both by state.

In [12]:
# Get all states represented in the dataset
states = sorted(set(
    state.strip().title()
    for row in conn.execute(
        'SELECT known_states FROM persons WHERE known_states IS NOT NULL'
    ).fetchall()
    for state in json.loads(row['known_states'] or '[]')
    if state.strip()
))

print(f"States in dataset ({len(states)}): {', '.join(states)}\n")

state_results = {}
for state in states:  # NOTE: no artificial limit — searches all states
    print(f"\n--- {state} ---")
    state_hits = []
    
    # Mug books
    query = f'"{state}" biographical history county mediatype:texts'
    mug_results = search_ia_metadata(query, rows=10, date_range=('1875', '1910'))
    if mug_results:
        print(f"  Mug books ({len(mug_results)} results):")
        for r in mug_results[:3]:
            print(f"    [{r['identifier']}] {r['title'][:60]} ({r['date']})")
        state_hits.extend(mug_results)
    time.sleep(IA_REQUEST_DELAY)
    
    # Press association proceedings
    query = f'"{state} press association" proceedings mediatype:texts'
    press_results = search_ia_metadata(query, rows=10, date_range=('1860', '1920'))
    if press_results:
        print(f"  Press association proceedings ({len(press_results)} results):")
        for r in press_results[:3]:
            print(f"    [{r['identifier']}] {r['title'][:60]} ({r['date']})")
        state_hits.extend(press_results)
    time.sleep(IA_REQUEST_DELAY)
    
    if not mug_results and not press_results:
        print('  (no results)')
    
    state_results[state] = state_hits

total_state_hits = sum(len(v) for v in state_results.values())
print(f"\n\nTotal state-level results: {total_state_hits}")
print('Review the above and add relevant IA identifiers to ia_corpus_manifest.json')

States in dataset (38): Alabama, Arizona, Arkansas, California, Connecticut, Dakota, Delaware, District Of Columbia, Georgia, Idaho, Illinois, Indiana, Iowa, Kansas, Kentucky, Louisiana, Maine, Maryland, Michigan, Minnesota, Mississippi, Missouri, Montana, Nebraska, Nevada, New York, Ohio, Oregon, Pennsylvania, South Carolina, Tennessee, Texas, Utah, Vermont, Virginia, Washington, West Virginia, Wisconsin


--- Alabama ---
  Mug books (8 results):
    [confederatemilit71evan] Confederate military history; a library of Confederate State (1899-01-01T00:00:00Z)
    [jeffersoncountyb00dubo] Jefferson County and Birmingham, Alabama; historical and bio (1887-01-01T00:00:00Z)
    [historyofconecuh00rile] History of Conecuh County, Alabama. Embracing a detailed rec (1881-01-01T00:00:00Z)

--- Arizona ---
  (no results)

--- Arkansas ---
  Mug books (10 results):
    [reminiscenthisto00unse] A reminiscent history of the Ozark region : comprising a con (1894-01-01T00:00:00Z)
    [confederatemili

## Append state-level results to manifest

Optionally run this cell to auto-append state-level results to the manifest. You should still review and prune manually.

In [13]:
# Reload manifest (in case you've edited it manually)
with open(MANIFEST_PATH) as f:
    manifest = json.load(f)

seen_ids = {item['identifier'] for item in manifest['items']}
added = 0

for state, hits in state_results.items():
    for r in hits[:5]:  # top 5 per state
        iid = r['identifier']
        if not iid or iid in seen_ids:
            continue
        seen_ids.add(iid)
        manifest['items'].append({
            'identifier': iid,
            'title': r['title'],
            'url': f'https://archive.org/details/{iid}',
            'persons_count': 0,
            'person_names_sample': '',
            'priority': 'medium',
            'notes': f'State source ({state}): {r["search_type"]}',
            'added_by': 'auto_step2_5_state',
        })
        added += 1

with open(MANIFEST_PATH, 'w') as f:
    json.dump(manifest, f, indent=2)

print(f'Added {added} state-level items to manifest.')
print(f'Total manifest items: {len(manifest["items"])}')
print(f'\nRemember to review ia_corpus_manifest.json before running Step 3a!')

Added 137 state-level items to manifest.
Total manifest items: 213

Remember to review ia_corpus_manifest.json before running Step 3a!
